# California Housing Price Prediction: Multivariate Regression Analysis

## Business Context
A real estate investment firm wants to build a predictive model for California housing prices.
This analysis identifies which factors — such as median income, average number of rooms, and
average occupancy — most strongly influence Median House Value.

We also test whether adding a non-linear (quadratic) term for income improves model accuracy.

## Objectives
- Identify significant predictors of house prices
- Assess estimate uncertainty using confidence intervals
- Compare a linear vs. quadratic model using Adjusted R²

## Setup: Import Libraries and Load Data

In [2]:
# CodeGrade step0

from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import os
import tarfile
import joblib
import statsmodels.api as sm
from sklearn.datasets._base import _pkl_filepath, get_data_home

# Process to get the data to be as it is in scikit-learn
archive_path = "cal_housing.tgz"
data_home = get_data_home(data_home=None)
if not os.path.exists(data_home):
    os.makedirs(data_home)
filepath = _pkl_filepath(data_home, 'cal_housing.pkz')

with tarfile.open(mode="r:gz", name=archive_path) as f:
    cal_housing = np.loadtxt(
        f.extractfile('CaliforniaHousing/cal_housing.data'),
        delimiter=',')
    columns_index = [8, 7, 2, 3, 4, 5, 6, 1, 0]
    cal_housing = cal_housing[:, columns_index]

    joblib.dump(cal_housing, filepath, compress=6)

# Load the dataset
california = fetch_california_housing(as_frame=True)
data = california.data
target = california.target

## Step 1: Fit the Multivariate Linear Regression Model

We select **MedInc**, **AveRooms**, and **AveOccup** as our predictor variables (X),
and **MedHouseVal** as our target (y).
A constant is added to X to account for the intercept term.
We fit the OLS model and report the R² value.

In [12]:
# CodeGrade step1

X = data[['MedInc', 'AveRooms', 'AveOccup']]
X = sm.add_constant(X)
y = target

mlr_model = sm.OLS(y, X).fit()
round(mlr_model.rsquared, 4)


0.4808

## Step 2: Extract P-Values

P-values tell us the statistical significance of each predictor.
We extract the first four p-values (intercept + three predictors) from the fitted model,
rounded to 5 decimal places.

In [6]:
# CodeGrade step2
p_values = mlr_model.pvalues
round(p_values.iloc[0], 5), round(p_values.iloc[1], 5), round(p_values.iloc[2], 5), round(p_values.iloc[3], 5)

(0.0, 0.0, 0.0, 0.0)

## Step 3: Identify Significant Predictors

We filter predictors whose p-value is **strictly less than α = 0.05**.
These are considered statistically significant in explaining median house value.
We report the shape of the resulting significant_predictors object.

In [9]:
# CodeGrade step3
significant_predictors = p_values[p_values < 0.05].index
significant_predictors = significant_predictors.drop('const')
significant_predictors.shape

(3,)

## Step 4: Compute 95% Confidence Intervals

Confidence intervals give us a range within which the true coefficient value is expected
to fall 95% of the time.
We extract specific values from the interval table using `.iloc[row, col]` indexing.

In [10]:
# CodeGrade step4
conf_intervals = mlr_model.conf_int(alpha=0.05)

round(conf_intervals.iloc[0, 0], 2), \
round(conf_intervals.iloc[0, 1], 2), \
round(conf_intervals.iloc[1, 0], 2), \
round(conf_intervals.iloc[1, 1], 2)

(0.58, 0.64, 0.43, 0.44)

## Step 5: Add a Quadratic Term (MedInc²)

To test whether the relationship between median income and house value is non-linear,
we add **MedInc_squared** (MedInc²) as an additional feature and refit the model.
We compare the new R² to the baseline model.

In [11]:
# CodeGrade step5
data['MedInc_squared'] = data['MedInc'] ** 2
X = data[['MedInc', 'MedInc_squared', 'AveRooms', 'AveOccup']]
X = sm.add_constant(X)

quad_model = sm.OLS(y, X).fit()
round(quad_model.rsquared, 4)

0.4858

## Step 6: Compare Adjusted R²

R² always increases when more predictors are added, even if they add no real value.
**Adjusted R²** penalizes for unnecessary complexity.
We compute and compare adjusted R² for both the base (linear) and quadratic models
to determine whether the added term genuinely improves fit.

In [13]:
# CodeGrade step6

adjusted_r2_base = mlr_model.rsquared_adj
adjusted_r2_quad = quad_model.rsquared_adj

round(adjusted_r2_base, 4), round(adjusted_r2_quad, 4)

(0.4807, 0.4857)